# Kepler Object Disposition Classification

A project by Alper Ozoner and Ali Utku Aydın. Compare logistic regression and k-nearest neighbors on the NASA Exoplanet Archive's Kepler Objects of Interest cumulative table ([DOI 10.26133/NEA4](https://doi.org/10.26133/NEA4)). Install the repository's requirements and run this notebook from the repository root. The first run downloads the public CSV.

The label is the catalog's published disposition. This walkthrough focuses on confirmed objects and false positives and leaves unresolved candidates out of the evaluation.

## 1. Load the data and inspect the available measurements

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display
from evaluate import FEATURES, download_data, evaluate, explore_clusters, held_out_split, load_data

data = Path('data/kepler-koi.csv')
if not data.exists():
    download_data(data)
features, labels, stars = load_data(data)
display(pd.Series(labels).map({0: 'False positive', 1: 'Confirmed'}).value_counts())
display(features[FEATURES].isna().mean().sort_values(ascending=False).rename('Missing fraction'))

## 2. Separate stars for training and testing

Multiple observations can share a Kepler star. Group by the star ID so a star never appears in two partitions. The features exclude the star ID, disposition score and post-vetting flags. Each fold fits median imputation and standardization on its own training rows.

In [ ]:
train, test = held_out_split(features, labels, stars)
assert set(stars[train]).isdisjoint(stars[test])
print(f'Training observations: {len(train):,} | test observations: {len(test):,}')

## 3. Compare models on the same holdout

Tune KNN using grouped cross-validation inside the training split. Then compare it with logistic regression and a majority baseline on the held-out stars. Macro F1 weights the two classes equally.

In [ ]:
output = Path('outputs')
scores = evaluate(features, labels, stars, output)
display(pd.DataFrame(scores['holdout']).T.round(3))
display(Image(filename=str(output / 'roc.png')))

## 4. Explore structure in the training data

Cluster a standardized sample of training observations and compare their silhouette scores. Cluster membership describes feature similarity and does not establish astronomical classes. The held-out model scores apply to established catalog entries in this split.

In [ ]:
clusters = explore_clusters(features, labels, stars)
display(pd.Series(clusters, name='Silhouette').rename_axis('Clusters').round(3))